# Synthetic E-Commerce Data Generator

**Portfolio use case:** Double-Date campaign performance analysis  
**Primary purpose:** Create a deterministic synthetic dataset for SQL and business-analysis practice.

This notebook:

- uses no confidential company data;
- generates fictional users, merchants, products, sessions, funnel events, orders, inventory, and returns;
- uses a fixed random seed so the generation logic is reproducible;
- exports 13 warehouse-ready tables as Parquet files;
- creates `synthetic_ecommerce.zip` as the permanent data snapshot.

BigQuery loading is intentionally handled in the separate
`02_load_or_restore_snapshot_to_bigquery.ipynb` notebook. Separating generation
from loading prevents the dataset from being regenerated when BigQuery Sandbox
tables expire.

> The archived Parquet files are the source of truth for the portfolio results.

## 1. Timeframe decision

The main portfolio window is **1 September–31 December 2024** so all four double-date events can be compared: 9.9, 10.10, 11.11, and 12.12.

However, that period is **not Q4**—Q4 is October–December. This notebook generates:

- **Data window:** 1 August–31 December 2024
- **Analysis window:** 1 September–31 December 2024
- **Purpose of August:** warm-up/baseline so September has a valid prior month and early-September analysis has pre-period context
- **Campaign windows:** 1–9 September, 1–10 October, 1–11 November, and 1–12 December
- **Peak day:** the double-date itself; retained separately from the full campaign window

Recommended portfolio wording: **“September–December 2024 campaign analysis, with August as the pre-period baseline.”**


## 2. Warehouse model and table grain

| Table | Grain | Main analytical use |
|---|---|---|
| `dim_date` | one row per calendar date | WoW, MoM, payday, campaign phase/intensity |
| `dim_users` | one row per registered synthetic user | cohort, loyalty, geography, acquisition |
| `dim_merchants` | one row per seller | merchant type, region, rating, commission |
| `dim_products` | one row per SKU | C1/C2 hierarchy, merchant, price, popularity |
| `dim_campaigns` | one row per double-date campaign | event window, traffic/CVR uplift, promo rate |
| `bridge_campaign_skus` | one row per campaign–SKU participation | featured assortment and promo participation |
| `fact_sessions` | one row per visit/session | traffic, visitors, device, channel |
| `fact_funnel_events` | one row per funnel event | session → view → cart → checkout → purchase |
| `fact_orders` | one row per merchant-level order | TPV, orders, payment, delivery, user/session |
| `fact_order_items` | one row per order line | quantity, SKU/category, discounts, profitability |
| `fact_inventory_daily` | one row per date–SKU | availability, viewable SKU, OOS, units sold |
| `fact_returns` | one row per returned order item | refund and return-rate analysis |

### Relationship logic

- `user_id` is nullable in traffic because anonymous sessions exist.
- `order_id` does **not** appear on sessions that fail before purchase.
- A completed funnel connects to an order through `session_id`.
- Product-level sales belong in `fact_order_items`, not in the order header.
- Promo costs are separated into seller discount, platform discount, voucher, cashback, and shipping subsidy.


## 3. Metric definitions used in this synthetic project

Internal acronyms vary by company. For portfolio use, define every metric explicitly instead of presenting unexplained proprietary terms.

| Metric | Synthetic definition |
|---|---|
| `gross_merchandise_value` | `list_price × quantity` |
| `customer_paid_tpv` | GMV − seller discount − platform discount − voucher; cashback is excluded because it is credited after purchase |
| `commission_revenue` | commission rate × value after seller discount |
| `gpbd` | commission revenue + service-fee revenue − payment fee |
| `gpad_tp` | GPBD − platform discount − voucher − cashback − shipping subsidy |
| `conversion_rate` | purchasing sessions ÷ all sessions |
| `viewable_sku_rate` | viewable SKU-date rows ÷ all active SKU-date rows |
| `availability_rate` | share of the day in which a SKU is purchasable |

These are **portfolio definitions**, not claims about Blibli's internal accounting definitions.


In [ ]:
# Colab setup. Run once.
%pip -q install pyarrow google-cloud-bigquery


In [ ]:
import os
import math
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

SEED = 42
rng = np.random.default_rng(SEED)

# Use environment variable SYNTH_PROFILE=test for a fast local smoke test.
PROFILE = os.getenv("SYNTH_PROFILE", "medium")

PROFILES = {
    "test":   {"n_users": 200,   "n_merchants": 8,   "n_skus": 30,   "base_sessions_per_day": 80},
    "small":  {"n_users": 10_000,"n_merchants": 40,  "n_skus": 250,  "base_sessions_per_day": 2_200},
    "medium": {"n_users": 30_000,"n_merchants": 80,  "n_skus": 600,  "base_sessions_per_day": 5_000},
    "large":  {"n_users": 80_000,"n_merchants": 150, "n_skus": 1_200,"base_sessions_per_day": 10_000},
}

if PROFILE not in PROFILES:
    raise ValueError(f"Unknown PROFILE={PROFILE!r}. Choose {list(PROFILES)}")

CFG = PROFILES[PROFILE]
DATA_START = pd.Timestamp("2024-08-01")
ANALYSIS_START = pd.Timestamp("2024-09-01")
DATA_END = pd.Timestamp("2024-12-31")

OUTPUT_DIR = Path("/content/synthetic_ecommerce") if Path("/content").exists() else Path("./synthetic_ecommerce")
EXPORT_CSV_TOO = False  # Parquet is recommended; turn on only if CSV is needed.

print("Profile:", PROFILE, CFG)
print("Data window:", DATA_START.date(), "to", DATA_END.date())
print("Analysis window:", ANALYSIS_START.date(), "to", DATA_END.date())


In [ ]:
# ---------- Shared reference data ----------
CATEGORY_CONFIG = pd.DataFrame([
    ("Sembako & Bumbu Masak", 48_000, 0.190, 0.010, 1.35),
    ("Minuman", 32_000, 0.205, 0.008, 1.30),
    ("Makanan Ringan", 27_000, 0.220, 0.008, 1.18),
    ("Daging Segar, Seafood & Makanan Beku", 78_000, 0.155, 0.020, 0.78),
    ("Buah & Sayur", 42_000, 0.170, 0.025, 0.72),
    ("Susu & Produk Olahan", 55_000, 0.185, 0.012, 1.02),
    ("Kebutuhan Ibu & Anak", 115_000, 0.145, 0.025, 0.88),
    ("Perawatan Rumah & Kebersihan", 69_000, 0.165, 0.012, 0.95),
    ("Kesehatan & Perawatan Diri", 82_000, 0.155, 0.018, 0.82),
    ("Kebutuhan Hewan Peliharaan", 74_000, 0.160, 0.015, 0.70),
], columns=["category_c2", "median_price", "base_atc_rate", "base_return_rate", "demand_weight"])

REGIONS = np.array(["Jabodetabek", "West Java", "Central Java", "East Java", "Sumatra", "Bali & Nusa Tenggara", "Kalimantan", "Sulawesi"])
REGION_P = np.array([0.34, 0.16, 0.12, 0.15, 0.10, 0.04, 0.05, 0.04])
CHANNELS = np.array(["organic", "paid_search", "paid_social", "affiliate", "direct", "crm"])
CHANNEL_P = np.array([0.28, 0.18, 0.17, 0.08, 0.21, 0.08])
DEVICES = np.array(["android_app", "ios_app", "mobile_web", "desktop_web"])
DEVICE_P = np.array([0.54, 0.16, 0.20, 0.10])

campaign_rows = [
    ("CAMP_0909", "9.9 Super Shopping Day", "2024-09-01", "2024-09-09", "2024-09-09", 1.40, 1.14, 0.030),
    ("CAMP_1010", "10.10 Brand Festival", "2024-10-01", "2024-10-10", "2024-10-10", 1.52, 1.18, 0.035),
    ("CAMP_1111", "11.11 Mega Sale", "2024-11-01", "2024-11-11", "2024-11-11", 1.74, 1.24, 0.040),
    ("CAMP_1212", "12.12 Year-End Sale", "2024-12-01", "2024-12-12", "2024-12-12", 2.15, 1.34, 0.050),
]
dim_campaigns = pd.DataFrame(campaign_rows, columns=[
    "campaign_id", "campaign_name", "start_date", "end_date", "event_date",
    "traffic_uplift", "cvr_uplift", "base_platform_discount_rate"
])
for c in ["start_date", "end_date", "event_date"]:
    dim_campaigns[c] = pd.to_datetime(dim_campaigns[c]).dt.date

dim_campaigns


In [ ]:
# ---------- Date dimension ----------
dates = pd.date_range(DATA_START, DATA_END, freq="D")
dim_date = pd.DataFrame({"date": dates})
dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = "Q" + dim_date["date"].dt.quarter.astype(str)
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name().str[:3]
dim_date["week_start"] = dim_date["date"] - pd.to_timedelta(dim_date["date"].dt.weekday, unit="D")
dim_date["day_of_week"] = dim_date["date"].dt.day_name()
dim_date["is_weekend"] = dim_date["date"].dt.weekday >= 5
dim_date["is_payday_period"] = dim_date["date"].dt.day >= 25
dim_date["is_analysis_window"] = dim_date["date"] >= ANALYSIS_START
dim_date["campaign_id"] = pd.NA
dim_date["is_campaign_window"] = False
dim_date["is_double_date"] = False
dim_date["campaign_day_number"] = pd.Series(pd.NA, index=dim_date.index, dtype="Int64")
dim_date["days_to_event"] = pd.Series(pd.NA, index=dim_date.index, dtype="Int64")
dim_date["campaign_phase"] = "non_campaign"
dim_date["campaign_intensity"] = 0.0

for row in dim_campaigns.itertuples(index=False):
    start, end, event = pd.Timestamp(row.start_date), pd.Timestamp(row.end_date), pd.Timestamp(row.event_date)
    mask = dim_date["date"].between(start, end)
    elapsed_days = (dim_date.loc[mask, "date"] - start).dt.days
    total_build_days = max((event - start).days, 1)
    progress = (elapsed_days / total_build_days).clip(0, 1)
    intensity = 0.25 + 0.75 * np.power(progress, 1.60)
    dim_date.loc[mask, "campaign_id"] = row.campaign_id
    dim_date.loc[mask, "is_campaign_window"] = True
    dim_date.loc[mask, "campaign_day_number"] = (elapsed_days + 1).astype(int)
    dim_date.loc[mask, "days_to_event"] = (event - dim_date.loc[mask, "date"]).dt.days.astype(int)
    dim_date.loc[mask & dim_date["date"].lt(event), "campaign_phase"] = "build_up"
    dim_date.loc[mask & dim_date["date"].le(start + pd.Timedelta(days=2)), "campaign_phase"] = "launch"
    dim_date.loc[mask, "campaign_intensity"] = intensity.to_numpy()
    dim_date.loc[dim_date["date"].eq(event), "campaign_phase"] = "peak_day"
    dim_date.loc[dim_date["date"].eq(event), "campaign_intensity"] = 1.0
    dim_date.loc[dim_date["date"].eq(event), "is_double_date"] = True

dim_date["date"] = dim_date["date"].dt.date
dim_date["week_start"] = dim_date["week_start"].dt.date
dim_date.head()


In [ ]:
# ---------- User dimension (no names, email addresses, phone numbers, or real PII) ----------
n_users = CFG["n_users"]
user_ids = np.array([f"U{i:07d}" for i in range(1, n_users + 1)])
loyalty = rng.choice(["new", "regular", "loyal", "vip"], size=n_users, p=[0.24, 0.48, 0.23, 0.05])

dim_users = pd.DataFrame({
    "user_id": user_ids,
    "registration_date": pd.to_datetime(rng.integers(pd.Timestamp("2022-01-01").value // 10**9,
                                                      pd.Timestamp("2024-07-31").value // 10**9,
                                                      size=n_users), unit="s").date,
    "age_band": rng.choice(["18-24", "25-34", "35-44", "45-54", "55+"], size=n_users,
                           p=[0.15, 0.39, 0.28, 0.13, 0.05]),
    "region": rng.choice(REGIONS, size=n_users, p=REGION_P),
    "city_tier": rng.choice(["Tier 1", "Tier 2", "Tier 3"], size=n_users, p=[0.46, 0.37, 0.17]),
    "acquisition_channel": rng.choice(CHANNELS, size=n_users, p=CHANNEL_P),
    "loyalty_tier": loyalty,
})

# Higher tiers create more repeat sessions.
user_activity_weight = pd.Series(loyalty).map({"new": 0.55, "regular": 1.0, "loyal": 1.9, "vip": 3.2}).to_numpy(float)
user_activity_weight /= user_activity_weight.sum()

dim_users.head()


In [ ]:
# ---------- Merchant and product dimensions ----------
n_merchants = CFG["n_merchants"]
merchant_ids = np.array([f"M{i:05d}" for i in range(1, n_merchants + 1)])
seller_type = rng.choice(["regular", "flagship", "official_store"], size=n_merchants, p=[0.60, 0.25, 0.15])

dim_merchants = pd.DataFrame({
    "merchant_id": merchant_ids,
    "merchant_name": [f"Synthetic Merchant {i:03d}" for i in range(1, n_merchants + 1)],
    "seller_type": seller_type,
    "merchant_region": rng.choice(REGIONS, size=n_merchants, p=REGION_P),
    "merchant_rating": np.round(np.clip(rng.normal(4.55, 0.24, n_merchants), 3.5, 5.0), 2),
    "fulfillment_type": rng.choice(["merchant_fulfilled", "platform_fulfilled"], size=n_merchants, p=[0.68, 0.32]),
})
dim_merchants["commission_rate"] = dim_merchants["seller_type"].map({
    "regular": 0.065, "flagship": 0.058, "official_store": 0.052
}).astype(float)

n_skus = CFG["n_skus"]
category_idx = rng.choice(len(CATEGORY_CONFIG), size=n_skus,
                          p=(CATEGORY_CONFIG["demand_weight"] / CATEGORY_CONFIG["demand_weight"].sum()).to_numpy())
chosen_categories = CATEGORY_CONFIG.iloc[category_idx].reset_index(drop=True)
sku_ids = np.array([f"SKU{i:06d}" for i in range(1, n_skus + 1)])
sku_merchant = rng.choice(merchant_ids, size=n_skus)
price_multiplier = rng.lognormal(mean=0.0, sigma=0.45, size=n_skus)

dim_products = pd.DataFrame({
    "sku_id": sku_ids,
    "sku_name": [f"Synthetic {chosen_categories.loc[i, 'category_c2']} Item {i+1:04d}" for i in range(n_skus)],
    "merchant_id": sku_merchant,
    "category_c1": "Bliblimart",
    "category_c2": chosen_categories["category_c2"],
    "brand_tier": rng.choice(["value", "mainstream", "premium"], size=n_skus, p=[0.30, 0.55, 0.15]),
    "list_price": np.round(np.maximum(5_000, chosen_categories["median_price"].to_numpy() * price_multiplier), -2),
    "base_atc_rate": chosen_categories["base_atc_rate"].to_numpy(),
    "base_return_rate": chosen_categories["base_return_rate"].to_numpy(),
    "popularity_score": rng.lognormal(mean=0.0, sigma=0.85, size=n_skus),
    "is_active": True,
})
dim_products = dim_products.merge(dim_merchants[["merchant_id", "commission_rate"]], on="merchant_id", how="left")
dim_products.head()


In [ ]:
# ---------- Campaign-to-SKU participation bridge ----------
bridge_parts = []
base_popularity = dim_products["popularity_score"].to_numpy(float)
base_popularity /= base_popularity.sum()

for idx, camp in dim_campaigns.iterrows():
    participation_share = [0.30, 0.36, 0.43, 0.52][idx]
    n_featured = max(1, int(n_skus * participation_share))
    selected = rng.choice(sku_ids, size=n_featured, replace=False, p=base_popularity)
    part = pd.DataFrame({
        "campaign_id": camp["campaign_id"],
        "sku_id": selected,
        "is_featured": rng.random(n_featured) < 0.28,
        "seller_discount_rate_plan": np.round(rng.uniform(0.03, 0.12, n_featured), 4),
        "platform_discount_rate_plan": np.round(
            camp["base_platform_discount_rate"] * rng.uniform(0.70, 1.25, n_featured), 4
        ),
    })
    bridge_parts.append(part)

bridge_campaign_skus = pd.concat(bridge_parts, ignore_index=True)
bridge_campaign_skus.head()


In [ ]:
# ---------- Provisional daily inventory/availability ----------
inventory_base = pd.MultiIndex.from_product(
    [pd.date_range(DATA_START, DATA_END, freq="D"), sku_ids],
    names=["snapshot_date", "sku_id"]
).to_frame(index=False)

inventory_base = inventory_base.merge(
    dim_products[["sku_id", "merchant_id", "popularity_score"]], on="sku_id", how="left"
)
inventory_base["stock_start"] = rng.poisson(
    np.clip(18 + 10 * np.log1p(inventory_base["popularity_score"]), 8, 100)
).astype(int)
inventory_base["replenishment_qty"] = rng.poisson(7, len(inventory_base)).astype(int)

campaign_date_map = dim_date.set_index("date")["campaign_id"]
campaign_intensity_map = dim_date.set_index("date")["campaign_intensity"]
inventory_base["campaign_id"] = inventory_base["snapshot_date"].dt.date.map(campaign_date_map)
inventory_base["campaign_intensity"] = inventory_base["snapshot_date"].dt.date.map(campaign_intensity_map).fillna(0.0)
max_campaign_pressure = inventory_base["campaign_id"].map({
    "CAMP_0909": 0.05, "CAMP_1010": 0.07, "CAMP_1111": 0.10, "CAMP_1212": 0.15
}).fillna(0.0).to_numpy(float)
campaign_pressure = max_campaign_pressure * inventory_base["campaign_intensity"].to_numpy(float)

availability = rng.beta(42, 1.6, len(inventory_base)) - campaign_pressure
availability -= np.where(inventory_base["stock_start"].to_numpy() == 0, 0.80, 0.0)
inventory_base["availability_rate"] = np.round(np.clip(availability, 0.0, 1.0), 4)
inventory_base["oos_minutes"] = np.round((1 - inventory_base["availability_rate"]) * 1440).astype(int)
inventory_base["is_viewable"] = (inventory_base["stock_start"] > 0) & (inventory_base["availability_rate"] >= 0.50)

# Funnel generation uses this lookup; units sold and stock end are reconciled after orders exist.
inventory_lookup = inventory_base[["snapshot_date", "sku_id", "merchant_id", "availability_rate", "is_viewable"]].copy()
inventory_lookup["snapshot_date"] = inventory_lookup["snapshot_date"].dt.date


In [ ]:
# ---------- Session generation with seasonality, payday, weekend, and campaign effects ----------
campaign_lookup = dim_campaigns.set_index("campaign_id").to_dict("index")
date_rows = dim_date.copy()

month_multiplier = {8: 0.92, 9: 1.00, 10: 1.05, 11: 1.14, 12: 1.28}
anon_pool_size = max(500, int(CFG["n_users"] * 0.45))
anon_ids = np.array([f"ANON{i:07d}" for i in range(1, anon_pool_size + 1)])

session_parts = []
session_counter = 1

for drow in date_rows.itertuples(index=False):
    date_value = pd.Timestamp(drow.date)
    multiplier = month_multiplier[date_value.month]
    if drow.is_weekend:
        multiplier *= 1.12
    if drow.is_payday_period:
        multiplier *= 1.10
    if pd.notna(drow.campaign_id):
        max_traffic_uplift = campaign_lookup[drow.campaign_id]["traffic_uplift"]
        multiplier *= 1 + (max_traffic_uplift - 1) * drow.campaign_intensity
        max_cvr_uplift = campaign_lookup[drow.campaign_id]["cvr_uplift"]
        campaign_cvr_multiplier = 1 + (max_cvr_uplift - 1) * drow.campaign_intensity
    else:
        campaign_cvr_multiplier = 1.0

    n = int(rng.poisson(CFG["base_sessions_per_day"] * multiplier))
    ids = np.array([f"S{i:010d}" for i in range(session_counter, session_counter + n)])
    session_counter += n

    logged_mask = rng.random(n) < 0.74
    selected_user_idx = rng.choice(n_users, size=n, replace=True, p=user_activity_weight)
    selected_users = user_ids[selected_user_idx]
    user_col = np.where(logged_mask, selected_users, None)
    visitor_col = np.where(logged_mask, np.char.replace(selected_users, "U", "V"), rng.choice(anon_ids, size=n))

    if pd.notna(drow.campaign_id):
        participation = set(bridge_campaign_skus.loc[
            bridge_campaign_skus["campaign_id"].eq(drow.campaign_id), "sku_id"
        ])
        sku_weights = dim_products["popularity_score"].to_numpy(float).copy()
        featured_mask = dim_products["sku_id"].isin(participation).to_numpy()
        sku_weights[featured_mask] *= 1.85
    else:
        sku_weights = dim_products["popularity_score"].to_numpy(float).copy()
    sku_weights /= sku_weights.sum()
    primary_sku = rng.choice(sku_ids, size=n, p=sku_weights)

    seconds = rng.integers(0, 24 * 60 * 60, size=n)
    started_at = date_value + pd.to_timedelta(seconds, unit="s")

    region_values = rng.choice(REGIONS, size=n, p=REGION_P)
    if logged_mask.any():
        region_values[logged_mask] = dim_users.iloc[selected_user_idx[logged_mask]]["region"].to_numpy()

    session_parts.append(pd.DataFrame({
        "session_id": ids,
        "visitor_id": visitor_col,
        "user_id": user_col,
        "started_at": started_at,
        "session_date": date_value.date(),
        "device_type": rng.choice(DEVICES, size=n, p=DEVICE_P),
        "traffic_channel": rng.choice(CHANNELS, size=n, p=CHANNEL_P),
        "region": region_values,
        "is_logged_in": logged_mask,
        "primary_sku_id": primary_sku,
        "campaign_id": drow.campaign_id if pd.notna(drow.campaign_id) else None,
        "campaign_phase": drow.campaign_phase,
        "campaign_intensity": float(drow.campaign_intensity),
        "campaign_cvr_multiplier": float(campaign_cvr_multiplier),
        "selected_user_idx": selected_user_idx,
    }))

sessions_work = pd.concat(session_parts, ignore_index=True)

sessions_work = sessions_work.merge(
    dim_products[["sku_id", "merchant_id", "category_c2", "base_atc_rate"]],
    left_on="primary_sku_id", right_on="sku_id", how="left"
).drop(columns="sku_id")
sessions_work = sessions_work.merge(
    inventory_lookup,
    left_on=["session_date", "primary_sku_id"],
    right_on=["snapshot_date", "sku_id"], how="left", suffixes=("", "_inventory")
).drop(columns=["snapshot_date", "sku_id", "merchant_id_inventory"])

print(f"Generated {len(sessions_work):,} sessions")
sessions_work.head()


In [ ]:
# ---------- Funnel progression ----------
n = len(sessions_work)
campaign_cvr = sessions_work["campaign_cvr_multiplier"].to_numpy(float)

loyalty_factor_by_user = pd.Series(dim_users["loyalty_tier"]).map({
    "new": 0.88, "regular": 1.00, "loyal": 1.14, "vip": 1.24
}).to_numpy(float)
loyalty_factor = np.where(
    sessions_work["is_logged_in"].to_numpy(),
    loyalty_factor_by_user[sessions_work["selected_user_idx"].to_numpy()],
    0.90,
)
channel_factor = sessions_work["traffic_channel"].map({
    "organic": 1.05, "paid_search": 1.02, "paid_social": 0.88,
    "affiliate": 0.94, "direct": 1.10, "crm": 1.18
}).to_numpy(float)
availability = sessions_work["availability_rate"].fillna(0).to_numpy(float)
viewable = sessions_work["is_viewable"].fillna(False).to_numpy(bool)

view_prob = np.where(viewable, 0.93, 0.35)
has_view = rng.random(n) < view_prob

atc_prob = np.clip(
    sessions_work["base_atc_rate"].to_numpy(float) * campaign_cvr * loyalty_factor * channel_factor * (0.55 + 0.45 * availability),
    0.02, 0.48
)
has_atc = has_view & (rng.random(n) < atc_prob)

checkout_prob = np.clip(0.59 * campaign_cvr * loyalty_factor, 0.30, 0.84)
has_checkout = has_atc & (rng.random(n) < checkout_prob)

purchase_prob = np.clip(0.70 * campaign_cvr * channel_factor * availability, 0.12, 0.93)
has_purchase = has_checkout & (rng.random(n) < purchase_prob) & sessions_work["is_logged_in"].to_numpy()

funnel_counts = pd.Series({
    "session_start": n,
    "product_view": int(has_view.sum()),
    "add_to_cart": int(has_atc.sum()),
    "begin_checkout": int(has_checkout.sum()),
    "purchase": int(has_purchase.sum()),
})
display(funnel_counts.to_frame("sessions"))
print("Overall CVR:", f"{has_purchase.mean():.2%}")


In [ ]:
# ---------- Event fact table ----------
def build_event_rows(mask, event_name, low_seconds, high_seconds, include_sku):
    idx = np.flatnonzero(mask)
    if len(idx) == 0:
        return pd.DataFrame()
    out = sessions_work.iloc[idx][["session_id", "user_id", "started_at", "session_date", "primary_sku_id"]].copy()
    out["event_name"] = event_name
    out["event_ts"] = out["started_at"] + pd.to_timedelta(
        rng.integers(low_seconds, high_seconds, size=len(out)), unit="s"
    )
    out["event_date"] = out["session_date"]
    out["sku_id"] = out["primary_sku_id"] if include_sku else None
    return out[["session_id", "user_id", "event_ts", "event_date", "event_name", "sku_id"]]

event_parts = [
    build_event_rows(np.ones(n, dtype=bool), "session_start", 0, 10, False),
    build_event_rows(has_view, "product_view", 10, 240, True),
    build_event_rows(has_atc, "add_to_cart", 120, 600, True),
    build_event_rows(has_checkout, "begin_checkout", 300, 1_200, True),
    build_event_rows(has_purchase, "purchase", 600, 2_400, True),
]
fact_funnel_events = pd.concat(event_parts, ignore_index=True)
fact_funnel_events.insert(0, "event_id", [f"E{i:012d}" for i in range(1, len(fact_funnel_events) + 1)])
fact_funnel_events = fact_funnel_events.sort_values(["session_id", "event_ts", "event_id"]).reset_index(drop=True)

fact_sessions = sessions_work[[
    "session_id", "visitor_id", "user_id", "started_at", "session_date",
    "device_type", "traffic_channel", "region", "is_logged_in", "campaign_id"
]].copy()

print(f"Generated {len(fact_funnel_events):,} funnel events")
fact_funnel_events.head()


In [ ]:
# ---------- Orders and order items ----------
purchase_sessions = sessions_work.loc[has_purchase].copy().reset_index(drop=True)
purchase_event_time = fact_funnel_events.loc[
    fact_funnel_events["event_name"].eq("purchase"), ["session_id", "event_ts"]
].rename(columns={"event_ts": "order_ts"})
purchase_sessions = purchase_sessions.merge(purchase_event_time, on="session_id", how="left")

n_orders = len(purchase_sessions)
purchase_sessions["order_id"] = [f"O{i:09d}" for i in range(1, n_orders + 1)]
purchase_sessions["order_date"] = purchase_sessions["order_ts"].dt.date
purchase_sessions["payment_method"] = rng.choice(
    ["e_wallet", "virtual_account", "credit_card", "debit_card", "paylater"],
    size=n_orders, p=[0.31, 0.26, 0.16, 0.10, 0.17]
)
purchase_sessions["delivery_type"] = rng.choice(
    ["regular", "same_day", "instant", "pickup"], size=n_orders, p=[0.61, 0.19, 0.14, 0.06]
)

items_per_order = rng.choice([1, 2, 3, 4], size=n_orders, p=[0.58, 0.28, 0.11, 0.03])
merchant_to_skus = dim_products.groupby("merchant_id")["sku_id"].apply(list).to_dict()

item_order_id = []
item_sku_id = []
for row, item_count in zip(purchase_sessions.itertuples(index=False), items_per_order):
    candidate_skus = merchant_to_skus[row.merchant_id]
    selected = [row.primary_sku_id]
    if item_count > 1:
        selected.extend(rng.choice(candidate_skus, size=item_count - 1, replace=True).tolist())
    item_order_id.extend([row.order_id] * item_count)
    item_sku_id.extend(selected)

fact_order_items = pd.DataFrame({"order_id": item_order_id, "sku_id": item_sku_id})
fact_order_items.insert(0, "order_item_id", [f"OI{i:010d}" for i in range(1, len(fact_order_items) + 1)])
fact_order_items = fact_order_items.merge(
    purchase_sessions[["order_id", "order_date", "campaign_id"]], on="order_id", how="left"
).merge(
    dim_products[["sku_id", "merchant_id", "category_c2", "list_price", "base_return_rate", "commission_rate"]],
    on="sku_id", how="left"
)

fact_order_items["quantity"] = rng.choice([1, 2, 3], size=len(fact_order_items), p=[0.78, 0.18, 0.04])
fact_order_items["gross_merchandise_value"] = fact_order_items["list_price"] * fact_order_items["quantity"]

campaign_sku_keys = set(zip(bridge_campaign_skus["campaign_id"], bridge_campaign_skus["sku_id"]))
is_campaign_participant = np.array([
    (camp, sku) in campaign_sku_keys if pd.notna(camp) else False
    for camp, sku in zip(fact_order_items["campaign_id"], fact_order_items["sku_id"])
])

seller_discount_rate = np.where(
    is_campaign_participant,
    rng.uniform(0.04, 0.14, len(fact_order_items)),
    np.where(rng.random(len(fact_order_items)) < 0.25, rng.uniform(0.01, 0.06, len(fact_order_items)), 0.0)
)
campaign_platform_rate = fact_order_items["campaign_id"].map(
    dim_campaigns.set_index("campaign_id")["base_platform_discount_rate"]
).fillna(0.0).to_numpy(float)
platform_discount_rate = np.where(
    is_campaign_participant,
    campaign_platform_rate * rng.uniform(0.70, 1.20, len(fact_order_items)),
    np.where(rng.random(len(fact_order_items)) < 0.10, rng.uniform(0.005, 0.020, len(fact_order_items)), 0.0)
)

order_voucher_rate = dict(zip(
    purchase_sessions["order_id"],
    np.where(rng.random(n_orders) < 0.22, rng.uniform(0.015, 0.05, n_orders), 0.0)
))
voucher_rate = fact_order_items["order_id"].map(order_voucher_rate).to_numpy(float)
cashback_rate = np.where(is_campaign_participant, rng.uniform(0.0, 0.025, len(fact_order_items)), 0.0)

order_shipping_subsidy = dict(zip(
    purchase_sessions["order_id"],
    np.where(rng.random(n_orders) < 0.38, rng.choice([5_000, 8_000, 10_000, 15_000], n_orders), 0)
))
order_item_count = fact_order_items.groupby("order_id")["order_item_id"].transform("count")
shipping_subsidy = fact_order_items["order_id"].map(order_shipping_subsidy).to_numpy(float) / order_item_count

gmv = fact_order_items["gross_merchandise_value"].to_numpy(float)
fact_order_items["seller_discount"] = np.round(gmv * seller_discount_rate, 2)
fact_order_items["platform_discount"] = np.round(gmv * platform_discount_rate, 2)
fact_order_items["voucher_cost"] = np.round(gmv * voucher_rate, 2)
fact_order_items["cashback_cost"] = np.round(gmv * cashback_rate, 2)
fact_order_items["shipping_subsidy"] = np.round(shipping_subsidy, 2)

fact_order_items["customer_paid_tpv"] = np.round(
    gmv - fact_order_items["seller_discount"] - fact_order_items["platform_discount"] - fact_order_items["voucher_cost"], 2
).clip(lower=0)
fact_order_items["commission_revenue"] = np.round(
    (gmv - fact_order_items["seller_discount"]) * fact_order_items["commission_rate"], 2
)
fact_order_items["service_fee_revenue"] = np.round(fact_order_items["customer_paid_tpv"] * 0.006, 2)
fact_order_items["payment_fee"] = np.round(fact_order_items["customer_paid_tpv"] * 0.012, 2)
fact_order_items["gpbd"] = np.round(
    fact_order_items["commission_revenue"] + fact_order_items["service_fee_revenue"] - fact_order_items["payment_fee"], 2
)
fact_order_items["gpad_tp"] = np.round(
    fact_order_items["gpbd"]
    - fact_order_items["platform_discount"]
    - fact_order_items["voucher_cost"]
    - fact_order_items["cashback_cost"]
    - fact_order_items["shipping_subsidy"], 2
)

order_agg = fact_order_items.groupby("order_id", as_index=False).agg(
    item_lines=("order_item_id", "count"),
    quantity=("quantity", "sum"),
    gross_merchandise_value=("gross_merchandise_value", "sum"),
    tpv=("customer_paid_tpv", "sum"),
    seller_discount=("seller_discount", "sum"),
    platform_discount=("platform_discount", "sum"),
    voucher_cost=("voucher_cost", "sum"),
    cashback_cost=("cashback_cost", "sum"),
    shipping_subsidy=("shipping_subsidy", "sum"),
    gpbd=("gpbd", "sum"),
    gpad_tp=("gpad_tp", "sum"),
)

fact_orders = purchase_sessions[[
    "order_id", "session_id", "user_id", "merchant_id", "order_ts", "order_date",
    "payment_method", "delivery_type", "region", "campaign_id"
]].merge(order_agg, on="order_id", how="left")
fact_orders["order_status"] = "COMPLETED"

print(f"Generated {len(fact_orders):,} orders and {len(fact_order_items):,} order items")
fact_orders.head()


In [ ]:
# ---------- Returns and inventory reconciliation ----------
return_mask = rng.random(len(fact_order_items)) < fact_order_items["base_return_rate"].to_numpy(float)
returned_items = fact_order_items.loc[return_mask, [
    "order_item_id", "order_id", "order_date", "customer_paid_tpv", "category_c2"
]].copy()

days_to_return = rng.integers(2, 22, len(returned_items))
returned_items["return_date"] = pd.to_datetime(returned_items["order_date"]) + pd.to_timedelta(days_to_return, unit="D")
returned_items = returned_items.loc[returned_items["return_date"] <= DATA_END].copy()
returned_items["return_date"] = returned_items["return_date"].dt.date
returned_items["return_reason"] = rng.choice(
    ["damaged", "wrong_item", "quality_issue", "changed_mind", "late_delivery"],
    size=len(returned_items), p=[0.24, 0.17, 0.25, 0.16, 0.18]
)
returned_items["refund_type"] = rng.choice(["full", "partial"], size=len(returned_items), p=[0.84, 0.16])
returned_items["refund_amount"] = np.round(
    returned_items["customer_paid_tpv"] * np.where(returned_items["refund_type"].eq("full"), 1.0, 0.5), 2
)
returned_items.insert(0, "return_id", [f"R{i:08d}" for i in range(1, len(returned_items) + 1)])
fact_returns = returned_items[[
    "return_id", "order_item_id", "order_id", "return_date", "return_reason", "refund_type", "refund_amount"
]].reset_index(drop=True)

return_order_counts = fact_returns.groupby("order_id")["order_item_id"].nunique()
order_line_counts = fact_order_items.groupby("order_id")["order_item_id"].nunique()
full_return_orders = return_order_counts[return_order_counts.eq(order_line_counts.reindex(return_order_counts.index))].index
partial_return_orders = return_order_counts.index.difference(full_return_orders)
fact_orders.loc[fact_orders["order_id"].isin(partial_return_orders), "order_status"] = "PARTIALLY_RETURNED"
fact_orders.loc[fact_orders["order_id"].isin(full_return_orders), "order_status"] = "RETURNED"

# Reconcile inventory sales and stock end with generated order lines.
units_sold = fact_order_items.groupby(["order_date", "sku_id"], as_index=False)["quantity"].sum().rename(
    columns={"order_date": "snapshot_date", "quantity": "units_sold"}
)
inventory_base["snapshot_date"] = inventory_base["snapshot_date"].dt.date
fact_inventory_daily = inventory_base.merge(units_sold, on=["snapshot_date", "sku_id"], how="left")
fact_inventory_daily["units_sold"] = fact_inventory_daily["units_sold"].fillna(0).astype(int)
minimum_replenishment = np.maximum(
    fact_inventory_daily["units_sold"] - fact_inventory_daily["stock_start"] + 2, 0
)
fact_inventory_daily["replenishment_qty"] = np.maximum(
    fact_inventory_daily["replenishment_qty"], minimum_replenishment
).astype(int)
fact_inventory_daily["stock_end"] = (
    fact_inventory_daily["stock_start"] + fact_inventory_daily["replenishment_qty"] - fact_inventory_daily["units_sold"]
).clip(lower=0).astype(int)
fact_inventory_daily = fact_inventory_daily[[
    "snapshot_date", "sku_id", "merchant_id", "campaign_id", "stock_start",
    "replenishment_qty", "units_sold", "stock_end", "availability_rate",
    "oos_minutes", "is_viewable"
]]

print(f"Generated {len(fact_returns):,} return records")


In [ ]:
# ---------- Clean warehouse-facing dtypes and assemble tables ----------
for df, date_cols in [
    (dim_users, ["registration_date"]),
    (dim_campaigns, ["start_date", "end_date", "event_date"]),
    (dim_date, ["date", "week_start"]),
    (fact_sessions, ["session_date"]),
    (fact_funnel_events, ["event_date"]),
    (fact_orders, ["order_date"]),
    (fact_order_items, ["order_date"]),
    (fact_inventory_daily, ["snapshot_date"]),
    (fact_returns, ["return_date"]),
]:
    for col in date_cols:
        df[col] = pd.to_datetime(df[col]).dt.date

tables = {
    "dim_date": dim_date,
    "dim_users": dim_users,
    "dim_merchants": dim_merchants,
    "dim_products": dim_products,
    "dim_campaigns": dim_campaigns,
    "bridge_campaign_skus": bridge_campaign_skus,
    "fact_sessions": fact_sessions,
    "fact_funnel_events": fact_funnel_events,
    "fact_orders": fact_orders,
    "fact_order_items": fact_order_items,
    "fact_inventory_daily": fact_inventory_daily,
    "fact_returns": fact_returns,
}

summary = pd.DataFrame([
    {"table_name": name, "rows": len(df), "columns": len(df.columns),
     "memory_mb": df.memory_usage(deep=True).sum() / 1024**2}
    for name, df in tables.items()
]).sort_values("rows", ascending=False)
summary_display = summary.copy()
summary_display["rows"] = summary_display["rows"].map(lambda x: f"{x:,.0f}")
summary_display["memory_mb"] = summary_display["memory_mb"].map(lambda x: f"{x:,.1f}")
display(summary_display)


In [ ]:
# ---------- Referential and business-rule validation ----------
def assert_unique(df, cols, table_name):
    assert not df.duplicated(cols).any(), f"Duplicate key in {table_name}: {cols}"

def assert_fk(child, child_col, parent, parent_col, label, allow_null=False):
    values = child[child_col]
    if allow_null:
        values = values.dropna()
    missing = set(values.unique()) - set(parent[parent_col].unique())
    assert not missing, f"Broken FK {label}: {list(missing)[:5]}"

assert_unique(dim_users, ["user_id"], "dim_users")
assert_unique(dim_merchants, ["merchant_id"], "dim_merchants")
assert_unique(dim_products, ["sku_id"], "dim_products")
assert_unique(fact_sessions, ["session_id"], "fact_sessions")
assert_unique(fact_funnel_events, ["event_id"], "fact_funnel_events")
assert_unique(fact_orders, ["order_id"], "fact_orders")
assert_unique(fact_order_items, ["order_item_id"], "fact_order_items")

assert_fk(dim_products, "merchant_id", dim_merchants, "merchant_id", "product→merchant")
assert_fk(fact_sessions, "user_id", dim_users, "user_id", "session→user", allow_null=True)
assert_fk(fact_funnel_events, "session_id", fact_sessions, "session_id", "event→session")
assert_fk(fact_orders, "session_id", fact_sessions, "session_id", "order→session")
assert_fk(fact_orders, "user_id", dim_users, "user_id", "order→user")
assert_fk(fact_order_items, "order_id", fact_orders, "order_id", "item→order")
assert_fk(fact_order_items, "sku_id", dim_products, "sku_id", "item→product")
assert_fk(fact_returns, "order_item_id", fact_order_items, "order_item_id", "return→item")

funnel_order = ["session_start", "product_view", "add_to_cart", "begin_checkout", "purchase"]
stage_counts = fact_funnel_events.groupby("event_name")["session_id"].nunique().reindex(funnel_order)
assert stage_counts.is_monotonic_decreasing, "Funnel counts must be non-increasing"
assert stage_counts["purchase"] == len(fact_orders), "One purchase event should map to one synthetic order"

tpv_recalc = (
    fact_order_items["gross_merchandise_value"]
    - fact_order_items["seller_discount"]
    - fact_order_items["platform_discount"]
    - fact_order_items["voucher_cost"]
)
assert np.allclose(tpv_recalc, fact_order_items["customer_paid_tpv"], atol=0.02)

gpad_recalc = (
    fact_order_items["gpbd"]
    - fact_order_items["platform_discount"]
    - fact_order_items["voucher_cost"]
    - fact_order_items["cashback_cost"]
    - fact_order_items["shipping_subsidy"]
)
assert np.allclose(gpad_recalc, fact_order_items["gpad_tp"], atol=0.03)
assert (fact_order_items["customer_paid_tpv"] >= 0).all()
assert (fact_inventory_daily["stock_end"] >= 0).all()

print("✅ All primary-key, foreign-key, funnel, monetary, and inventory validations passed.")
display(stage_counts.to_frame("unique_sessions"))


In [ ]:
# ---------- Sanity check: synthetic campaign patterns should be visible ----------
daily_sessions = fact_sessions.groupby(["session_date", "campaign_id"], dropna=False).agg(
    sessions=("session_id", "nunique"),
    visitors=("visitor_id", "nunique")
).reset_index()

daily_orders = fact_orders.groupby(["order_date", "campaign_id"], dropna=False).agg(
    orders=("order_id", "nunique"),
    tpv=("tpv", "sum"),
    gpad_tp=("gpad_tp", "sum")
).reset_index()

daily_kpi = daily_sessions.merge(
    daily_orders,
    left_on=["session_date", "campaign_id"],
    right_on=["order_date", "campaign_id"],
    how="left"
)
daily_kpi[["orders", "tpv", "gpad_tp"]] = daily_kpi[["orders", "tpv", "gpad_tp"]].fillna(0)
daily_kpi["cvr"] = daily_kpi["orders"] / daily_kpi["sessions"]

campaign_summary = daily_kpi.groupby("campaign_id", dropna=False).agg(
    avg_daily_sessions=("sessions", "mean"),
    avg_daily_orders=("orders", "mean"),
    avg_daily_tpv=("tpv", "mean"),
    avg_cvr=("cvr", "mean"),
    total_gpad_tp=("gpad_tp", "sum"),
).reset_index()
display(campaign_summary)


In [ ]:
# ---------- Data dictionary and export ----------
table_descriptions = {
    "dim_date": "Calendar dimension with comparison and campaign flags",
    "dim_users": "Registered synthetic users without direct PII",
    "dim_merchants": "Synthetic marketplace sellers",
    "dim_products": "Synthetic SKU master and category hierarchy",
    "dim_campaigns": "Double-date event definitions and uplift parameters",
    "bridge_campaign_skus": "SKU participation in campaigns",
    "fact_sessions": "Session-level traffic fact",
    "fact_funnel_events": "Event-level funnel fact",
    "fact_orders": "Merchant-level order header and aggregated economics",
    "fact_order_items": "Line-level sales, promo, and profitability",
    "fact_inventory_daily": "Daily SKU availability and inventory snapshot",
    "fact_returns": "Returned item and refund records",
}

dictionary_rows = []
for table_name, df in tables.items():
    for position, col in enumerate(df.columns, 1):
        dictionary_rows.append({
            "table_name": table_name,
            "table_description": table_descriptions[table_name],
            "column_position": position,
            "column_name": col,
            "pandas_dtype": str(df[col].dtype),
            "nullable": bool(df[col].isna().any()),
        })
data_dictionary = pd.DataFrame(dictionary_rows)
tables["data_dictionary"] = data_dictionary

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for table_name, df in tables.items():
    df.to_parquet(OUTPUT_DIR / f"{table_name}.parquet", index=False)
    if EXPORT_CSV_TOO:
        df.to_csv(OUTPUT_DIR / f"{table_name}.csv.gz", index=False, compression="gzip")

archive_path = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print("Saved files to:", OUTPUT_DIR.resolve())
print("ZIP archive:", archive_path)
display(data_dictionary.head(20))


## 4. Load the archived snapshot into BigQuery

This generator intentionally stops after validation and Parquet export.

To load the exact snapshot into BigQuery:

1. Save or download the generated `synthetic_ecommerce.zip`.
2. Open `02_load_or_restore_snapshot_to_bigquery.ipynb`.
3. Set your Google Cloud project, dataset, location, and Drive folder.
4. Run that notebook to create 13 non-partitioned BigQuery tables.

The same loader can be run again when BigQuery Sandbox tables expire. It reloads
the archived Parquet snapshot and does not generate new values.

## 5. Starter SQL challenges for the portfolio

After loading the tables, avoid beginning with charts. First prove the data model and business logic through SQL.

### A. Compare double-date campaigns against a seven-day baseline

```sql
WITH daily_performance AS (
  SELECT
    o.order_date,
    o.campaign_id,
    COUNT(DISTINCT o.order_id) AS orders,
    SUM(o.tpv) AS tpv,
    SUM(o.gpad_tp) AS gpad_tp
  FROM `PROJECT.synthetic_ecommerce_portfolio.fact_orders` o
  GROUP BY 1, 2
),
campaign_window AS (
  SELECT
    c.campaign_id,
    c.campaign_name,
    AVG(IF(d.order_date BETWEEN c.start_date AND c.end_date, d.tpv, NULL)) AS campaign_daily_tpv,
    AVG(IF(d.order_date BETWEEN DATE_SUB(c.start_date, INTERVAL 7 DAY)
                            AND DATE_SUB(c.start_date, INTERVAL 1 DAY), d.tpv, NULL)) AS baseline_daily_tpv
  FROM `PROJECT.synthetic_ecommerce_portfolio.dim_campaigns` c
  CROSS JOIN daily_performance d
  GROUP BY 1, 2
)
SELECT *, SAFE_DIVIDE(campaign_daily_tpv, baseline_daily_tpv) - 1 AS tpv_uplift
FROM campaign_window
ORDER BY campaign_id;
```

### B. Build the funnel by channel using CTEs and conditional aggregation

```sql
WITH session_stage AS (
  SELECT
    s.session_id,
    s.traffic_channel,
    MAX(e.event_name = 'product_view') AS viewed,
    MAX(e.event_name = 'add_to_cart') AS added_to_cart,
    MAX(e.event_name = 'begin_checkout') AS checked_out,
    MAX(e.event_name = 'purchase') AS purchased
  FROM `PROJECT.synthetic_ecommerce_portfolio.fact_sessions` s
  LEFT JOIN `PROJECT.synthetic_ecommerce_portfolio.fact_funnel_events` e
    USING (session_id)
  GROUP BY 1, 2
)
SELECT
  traffic_channel,
  COUNT(*) AS sessions,
  COUNTIF(viewed) AS product_view_sessions,
  COUNTIF(added_to_cart) AS add_to_cart_sessions,
  COUNTIF(checked_out) AS checkout_sessions,
  COUNTIF(purchased) AS purchase_sessions,
  SAFE_DIVIDE(COUNTIF(purchased), COUNT(*)) AS cvr
FROM session_stage
GROUP BY 1
ORDER BY sessions DESC;
```

### C. Find category growth that destroys profitability

```sql
WITH monthly_category AS (
  SELECT
    DATE_TRUNC(order_date, MONTH) AS month,
    category_c2,
    SUM(customer_paid_tpv) AS tpv,
    SUM(gpad_tp) AS gpad_tp,
    SAFE_DIVIDE(SUM(gpad_tp), SUM(customer_paid_tpv)) AS gpad_tp_margin
  FROM `PROJECT.synthetic_ecommerce_portfolio.fact_order_items`
  GROUP BY 1, 2
)
SELECT
  *,
  LAG(tpv) OVER (PARTITION BY category_c2 ORDER BY month) AS prior_month_tpv,
  SAFE_DIVIDE(tpv, LAG(tpv) OVER (PARTITION BY category_c2 ORDER BY month)) - 1 AS mom_tpv_growth
FROM monthly_category
ORDER BY month, mom_tpv_growth DESC;
```


## 6. Suggested analysis order

1. Validate traffic, funnel, sales, and profitability totals.
2. Compare double-date campaign windows versus seven-day baselines.
3. Decompose TPV through Sessions × CVR × AOV.
4. Drill down by C2, merchant, SKU, channel, device, and user segment.
5. Add operational context from availability, viewable SKU, and OOS.
6. Separate **business performance observed** from **analytical impact or recommendation**.

### Public references

- Blibli publicly describes BlibliMart as covering daily essentials, groceries/fresh food, household supplies, and mother-and-child products: https://www.blibli.com/c1/bliblimart/53400
- Google Cloud documents `pandas-gbq` and `google-cloud-bigquery` for analysis and DataFrame uploads: https://cloud.google.com/bigquery/docs/python-libraries
- BigQuery DataFrame loading sample: https://cloud.google.com/bigquery/docs/samples/bigquery-load-table-dataframe
